# 04 - Transformer (D)

In [1]:
import sys
sys.path.append("..")

#from src.config import *
#from src.data import load_split
#from src.cleaning import clean_text
#from src.evaluate import evaluate, save_results

In [3]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0))

x = torch.rand(1000, 1000, device="cuda")
y = x @ x
print("Result:", y.sum().item())

PyTorch version: 2.15.0+cu132
CUDA available: True
GPU name: NVIDIA GeForce RTX 5060 Ti
Result: 250235840.0


In [4]:
import pandas as pd

DATA_DIR = "../data/raw"

def load_raw_split(split):
    """Read one split ("train", "val" or "test") of TweetEval irony into a DataFrame."""
    with open(f"{DATA_DIR}/{split}_text.txt", encoding="utf-8") as f:
        texts = f.read().splitlines()

    with open(f"{DATA_DIR}/{split}_labels.txt", encoding="utf-8") as f:
        labels = [int(line) for line in f.read().splitlines()]

    assert len(texts) == len(labels), f"{split}: {len(texts)} texts but {len(labels)} labels"

    return pd.DataFrame({"text": texts, "label": labels})


train_df = load_raw_split("train")
val_df = load_raw_split("val")
test_df = load_raw_split("test")

print("train:", train_df.shape)
print("val:  ", val_df.shape)
print("test: ", test_df.shape)

train: (2862, 2)
val:   (955, 2)
test:  (784, 2)


In [5]:
train_df.head()

,text,label
0,seeing ppl walking w/ crutches makes me really...,1
1,"look for the girl with the broken smile, ask h...",0
2,Now I remember why I buy books online @user #s...,1
3,@user @user So is he banded from wearing the c...,1
4,Just found out there are Etch A Sketch apps. ...,1


In [13]:
corrupted = train_df["text"].str.contains("\t")

print("Corrupted rows found:", corrupted.sum())
print(train_df.loc[corrupted, "text"].str.len())

train_df = train_df[~corrupted].reset_index(drop=True)
print("train after removal:", train_df.shape)

Corrupted rows found: 3
1295    1000
1753     329
2773     249
Name: text, dtype: int64
train after removal: (2859, 2)


In [6]:
from datasets import Dataset, DatasetDict

ds = DatasetDict({
    "train": Dataset.from_pandas(train_df),
    "validation": Dataset.from_pandas(val_df),
    "test": Dataset.from_pandas(test_df),
})

print(ds)
print(ds["train"][0])

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 2862
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 955
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 784
    })
})
{'text': 'seeing ppl walking w/ crutches makes me really excited for the next 3 weeks of my life  ', 'label': 1}


In [8]:
from transformers import AutoTokenizer

MODEL_NAME = "distilbert/distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

example = ds["train"][0]["text"]
encoded = tokenizer(example)

print("Tweet:    ", example)
print("input_ids:", encoded["input_ids"])
print("tokens:   ", tokenizer.convert_ids_to_tokens(encoded["input_ids"]))
print("attention:", encoded["attention_mask"])

Tweet:     seeing ppl walking w/ crutches makes me really excited for the next 3 weeks of my life  
input_ids: [101, 3773, 4903, 2140, 3788, 1059, 1013, 13675, 4904, 8376, 3084, 2033, 2428, 7568, 2005, 1996, 2279, 1017, 3134, 1997, 2026, 2166, 102]
tokens:    ['[CLS]', 'seeing', 'pp', '##l', 'walking', 'w', '/', 'cr', '##ut', '##ches', 'makes', 'me', 'really', 'excited', 'for', 'the', 'next', '3', 'weeks', 'of', 'my', 'life', '[SEP]']
attention: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [10]:
roberta_tokenizer = AutoTokenizer.from_pretrained("FacebookAI/roberta-base")

test_tweet = "Oh GREAT, another Monday 🙂"

for name, tok in [("DistilBERT", tokenizer), ("RoBERTa", roberta_tokenizer)]:
    ids = tok(test_tweet)["input_ids"]
    print(f"{name:10} -> {tok.convert_ids_to_tokens(ids)}")

DistilBERT -> ['[CLS]', 'oh', 'great', ',', 'another', 'monday', '[UNK]', '[SEP]']
RoBERTa    -> ['<s>', 'Oh', 'ĠGREAT', ',', 'Ġanother', 'ĠMonday', 'ĠðŁĻĤ', '</s>']


In [11]:
token_lengths = [len(tokenizer(text)["input_ids"]) for text in train_df["text"]]

lengths = pd.Series(token_lengths)
print(lengths.describe())
print("99th percentile:", lengths.quantile(0.99))

count    2862.000000
mean       22.868623
std        10.529508
min         3.000000
25%        15.000000
50%        22.000000
75%        30.000000
max       294.000000
dtype: float64
99th percentile: 44.0


In [12]:
print(train_df.loc[lengths.idxmax(), "text"])

@user @user Aaaaaamd what time is your bedtime? >better  tweet Dad at night from my house. I don't want to get in trouble!3029	1	Can't wait till my birthday 3030	0	Clovis Cartoon, Tuesday, December 09, 2014 #jamaicaobserverandroidapp via @user a real McCoy3031	1	Just now layin down to go to bed.|College is great guys. It doesn't stress you out or keep you up at night at all. Or make you cry. 3032	0	Pretty sure I just sat behind @user in the McDonald's drive through and peeked to see movie what was playing 😂3033	0	@SenRandPaul #ChrisChristy will  do well among republican voters.#JebBush, I think he could do well, democrats don't like him.3034	1	nights like tonight are the only reason I'm able to carry on to the weekend #SUCHAGOODWEEK 3035	1	wow I love how frizzy my hair gets 3036	0	@dbrauer #GoodMorning! @user @user @user @user them roulettes me there heavy3038	1	@NESRetku Joke of the year congrats! #Kuivaa  :/3039	1	@gilester haven't you used the don't you know who I am" line yet....  